# soundai — クイックスタート

このノートブックでは次の3ステップを試します。

1. `data/samples/` の音源をスキャンして CLAP embedding を生成
2. 自然言語クエリで検索
3. 音源ファイルで類似検索

> **前提**: `data/samples/` に WAV / AIFF / MP3 ファイルを配置してください。

In [2]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from tqdm.notebook import tqdm

from soundai.embed import CLAPEmbedder, SUPPORTED_EXTENSIONS
from soundai.index import EmbeddingIndex
from soundai.search import SampleSearcher

SAMPLES_DIR = Path("../data/samples")
INDEX_DIR   = Path("../data/embeddings")

## 1. 音源をインデックスに登録

CLI からも実行できます:
```bash
uv run python scripts/ingest.py
```

In [6]:
# 音源ファイルを収集
audio_files = [
    p for p in sorted(SAMPLES_DIR.rglob("*"))
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
]
print(f"見つかったファイル数: {len(audio_files)}")
audio_files[:5]

見つかったファイル数: 3090


[PosixPath('../data/samples/WatusiSamplesSmall_20190730/101 Techno Kicks/SM White Label –\xa0Analogue Techno/at_kick_1.wav'),
 PosixPath('../data/samples/WatusiSamplesSmall_20190730/101 Techno Kicks/SM White Label –\xa0Analogue Techno/at_kick_10.wav'),
 PosixPath('../data/samples/WatusiSamplesSmall_20190730/101 Techno Kicks/SM White Label –\xa0Analogue Techno/at_kick_11.wav'),
 PosixPath('../data/samples/WatusiSamplesSmall_20190730/101 Techno Kicks/SM White Label –\xa0Analogue Techno/at_kick_2.wav'),
 PosixPath('../data/samples/WatusiSamplesSmall_20190730/101 Techno Kicks/SM White Label –\xa0Analogue Techno/at_kick_3.wav')]

In [8]:
# Embedding 生成 (初回は CLAP モデルのダウンロードが走ります ~400MB)
BATCH_SIZE = 32

embedder = CLAPEmbedder()
index    = EmbeddingIndex()

for i in tqdm(range(0, len(audio_files), BATCH_SIZE)):
    batch = audio_files[i : i + BATCH_SIZE]
    embs  = embedder.embed_files(batch)
    if embs.shape[0] > 0:
        index.add(embs, [str(p) for p in batch[:embs.shape[0]]])

index.save(INDEX_DIR)
print(f"インデックス完成: {index.size} 件")

  0%|          | 0/97 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the specified checkpoint /Users/soichiro.inatani/src/soundai/.venv/lib/python3.11/site-packages/laion_clap/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/soichiro.inatani/src/soundai/.venv/lib/python3.11/site-packages/laion_clap/music_audioset_epoch_15_esc_90.14.pt'

## 2. 自然言語で検索

In [4]:
searcher = SampleSearcher(index_dir=INDEX_DIR)

query = "deep 808 bass kick"   # ← ここを自由に変えてみてください

results = searcher.search_by_text(query, k=10)
searcher.pretty_print(results)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the specified checkpoint /Users/soichiro.inatani/src/soundai/.venv/lib/python3.11/site-packages/laion_clap/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_bra

## 3. 音源ファイルで類似検索

In [ ]:
# 手持ちのファイルパスを指定してください
example_file = audio_files[0] if audio_files else None

if example_file:
    print(f"クエリ: {example_file.name}")
    results = searcher.search_by_file(example_file, k=10)
    searcher.pretty_print(results)
else:
    print("data/samples/ に音源ファイルを配置してください")

## 4. (ボーナス) UMAP で埋め込み空間を可視化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap

# インデックスから全 embedding を取り出す
n = index.size
all_embs = np.zeros((n, index.dim), dtype="float32")
index._index.reconstruct_n(0, n, all_embs)

reducer = umap.UMAP(n_components=2, random_state=42)
coords  = reducer.fit_transform(all_embs)

labels = [Path(p).parent.name for p in index._paths]   # フォルダ名をラベルに

plt.figure(figsize=(10, 8))
unique_labels = list(set(labels))
cmap = plt.cm.get_cmap("tab20", len(unique_labels))
for i, lbl in enumerate(unique_labels):
    mask = [l == lbl for l in labels]
    plt.scatter(coords[mask, 0], coords[mask, 1],
                label=lbl, s=20, alpha=0.7, color=cmap(i))
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.title("Audio Embedding Space (UMAP)")
plt.tight_layout()
plt.show()